# 1. What you'll learn

        - connect covariance eigenvectors to maximum-variance directions
- read explained variance and loadings
- reconstruct handwritten digits as k changes

        **The one question this notebook answers:** How can orthogonal directions preserve most variation, and what information is lost when you keep only k?

# 2. Setup

This lesson keeps principal component analysis small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

Scikit-learn's 8×8 handwritten digits provide 1,797 rows and 64 pixel-intensity columns. Labels are used only to colour projections, never to fit PCA.

**Small example:** For five centered 2D points lying near y=2x, the first eigenvector points along that diagonal. Projecting onto it keeps spread along the line and discards small perpendicular noise.

In [ ]:
digits=load_digits();X=digits.data;y=digits.target;print("Shape:",X.shape,"classes:",np.bincount(y).tolist(),"pixel range:",(X.min(),X.max()));print("One row reshaped:",X[0].reshape(8,8).astype(int))

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].imshow(X[0].reshape(8,8),cmap="gray_r");axes[0].set_title(f"One 8×8 digit: {y[0]}");axes[0].axis("off")
axes[1].hist(X.std(0),bins=20,color="#176b66");axes[1].set(title="Pixel variability",xlabel="standard deviation")
sns.heatmap(np.corrcoef(X[:,[18,19,20,26,27,28]].T),vmin=-1,vmax=1,cmap="vlag",annot=True,fmt=".1f",ax=axes[2]);axes[2].set_title("Nearby pixels correlate");plt.tight_layout();plt.show()

**Takeaway:** Each digit is a 64-dimensional point whose spatial structure is hidden in a flat row.

**Takeaway:** Many border pixels barely vary and cannot explain much covariance.

**Takeaway:** Neighbouring active pixels co-vary, creating compressible directions.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEED);scaler=StandardScaler().fit(X_train);A=scaler.transform(X_train);B=scaler.transform(X_test);print("Train/test:",A.shape,B.shape,"mean abs train mean:",abs(A.mean(0)).mean().round(6))

# 6. Train

        Training turns the assumptions behind principal component analysis into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Center and scale training columns.
2. Form directions that diagonalize the covariance matrix.
3. Order directions by eigenvalue from largest variance down.
4. Project rows onto the first k orthogonal directions.

In [ ]:
full=PCA(random_state=SEED).fit(A);cum=np.cumsum(full.explained_variance_ratio_);k95=int(np.searchsorted(cum,.95)+1);pca=PCA(n_components=k95,random_state=SEED).fit(A);Z=pca.transform(B);recon=scaler.inverse_transform(pca.inverse_transform(Z));print("Components for 95% variance:",k95,"latent shape:",Z.shape)

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
ks=[2,5,10,20,40,64];errors=[];recons={}
for k in ks:
 q=PCA(n_components=k,random_state=SEED).fit(A);r=scaler.inverse_transform(q.inverse_transform(q.transform(B)));errors.append(mean_squared_error(X_test,r)**.5);recons[k]=r
baseline=np.repeat(X_train.mean(0)[None,:],len(X_test),axis=0);print(f"k={k95} reconstruction RMSE={mean_squared_error(X_test,recon)**.5:.3f}; mean-image baseline={mean_squared_error(X_test,baseline)**.5:.3f}")
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].plot(range(1,65),cum);axes[0].axhline(.95,ls="--",color="black");axes[0].axvline(k95,ls="--",color="black");axes[0].set(title="Cumulative explained variance",xlabel="components")
axes[1].plot(ks,errors,marker="o",color="#d38b45");axes[1].set(title="Reconstruction error",xlabel="k",ylabel="pixel RMSE")
strip=np.hstack([X_test[0].reshape(8,8),recons[2][0].reshape(8,8),recons[10][0].reshape(8,8),recons[40][0].reshape(8,8)]);axes[2].imshow(strip,cmap="gray_r");axes[2].set_title("Original | k=2 | k=10 | k=40");axes[2].axis("off");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show()

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: PCA optimizes squared reconstruction and is therefore sensitive to scale and extreme outliers; high variance need not be predictive. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
rng=np.random.default_rng(SEED);outlier=X_train.copy();outlier[:8]+=rng.normal(0,80,size=(8,64));bad_scaler=StandardScaler().fit(outlier);bad=PCA(n_components=k95,random_state=SEED).fit(bad_scaler.transform(outlier));bad_recon=bad_scaler.inverse_transform(bad.inverse_transform(bad.transform(bad_scaler.transform(X_test))))
print(f"Clean PCA RMSE={mean_squared_error(X_test,recon)**.5:.3f}; with extreme training outliers={mean_squared_error(X_test,bad_recon)**.5:.3f}")

**Use this when…** linear correlated features contain redundant variation and compression or visualization is useful.

        **Don't use this when…** important structure is nonlinear, variance is not relevance, outliers dominate, or original-feature semantics must remain intact.

        ## Try this

        1. Skip scaling and identify pixels that dominate.
2. Whiten components and inspect their variances.
3. Train a classifier on 5, 15, and 40 components.